In [ ]:
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# Decision Tree Classifier for Image Segmentation
# ==========================================

def segment_image_dt(image_path, ref_segmentation_path=None):
    # 1. Load image and reference segmentation
    print(f"Loading image from {image_path}...")
    img = cv2.imread(image_path)
    if img is None:
        print("Image not found. Creating dummy data for demonstration.")
        # Create a dummy image and reference segmentation if not found
        img = np.random.randint(0, 255, (160, 109, 3), dtype=np.uint8)
        y = np.random.randint(0, 7, (160 * 109))
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Load reference segmentation labels here if available
        # y = cv2.imread(ref_segmentation_path, cv2.IMREAD_GRAYSCALE).ravel()
        # Fallback to random classes if ref is missing
        y = np.random.randint(0, 7, (img.shape[0] * img.shape[1]))
    
    h, w, _ = img.shape
    
    # 2. Extract RGB and normalized X, Y features
    print("Extracting features...")
    yy, xx = np.mgrid[0:h, 0:w]
    X = np.column_stack([img.reshape(-1,3)/255.0, xx.ravel()/w, yy.ravel()/h])
    
    # 3. Split into training and testing data
    print("Splitting data...")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 4. Train DecisionTreeClassifier
    print("Training Decision Tree...")
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(Xtr, ytr)
    
    # 5. Predict and calculate accuracy
    print("Evaluating...")
    pred = model.predict(Xte)
    acc = accuracy_score(yte, pred)
    
    print(f"Image Size = {w} x {h} pixels")
    print(f"Training samples = {len(Xtr)}")
    print(f"Testing samples = {len(Xte)}")
    print(f"Accuracy = {acc * 100:.2f}%")
    
    return model, acc

if __name__ == '__main__':
    # Update path to your image
    IMAGE_PATH = "sample_image.jpg"
    segment_image_dt(IMAGE_PATH)
